# Ridge Regression: From First Principles

## 1. Why Add Something to Linear Regression?

Imagine estimating house prices from floor area, bedroom count, and several measurements of living space. Ordinary Linear Regression can combine these features into a useful prediction. But floor area and living space may contain almost the same information. The training data may not tell us clearly how much credit each feature deserves.

One fitted model might assign a large positive coefficient to one measurement and a large negative coefficient to the other. Their contributions nearly cancel for the training houses. A small change in the data can then produce very different coefficients, even when training predictions remain similar. This is **instability**: the fitted model depends too strongly on the particular sample we happened to collect.

Ridge Regression keeps the linear prediction equation and changes how its coefficients are learned. It balances fitting the observed targets against using large coefficients. This added preference is called **regularization**.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Accept some extra training error when it buys a more stable set of coefficients. Whether that improves future predictions must be checked on unseen data.</div>

We will first understand the prediction rule, then the penalty, then fit a small model by hand. Finally, we will connect the mathematics to the code companion and a reliable evaluation workflow.

## 2. What Does Ridge Receive and Produce?

Ridge is a **supervised regression** method. During training, it receives rows of numerical features and their known numerical targets. For a new row, it returns a numerical prediction, such as an estimated house price. It does not directly produce a class label or a class probability.

Let $n$ be the number of training rows and $p$ the number of features. For row $i$:

$$
\hat y_i=b+\sum_{j=1}^{p}w_jx_{ij}.
$$

| Symbol | Meaning | House-price example |
|---|---|---|
| $x_{ij}$ | Feature $j$ for row $i$ | Floor area of house $i$ |
| $y_i$ | Observed target | Recorded sale price |
| $\hat y_i$ | Predicted target | Estimated sale price |
| $w_j$ | Learned feature coefficient | Change in prediction per unit of area, holding other features fixed |
| $b$ | Learned intercept | Prediction when every represented feature is zero |

A **residual** is $e_i=y_i-\hat y_i$: the part of the observed target the model did not predict. A positive residual means the model predicted too low.

The coefficients and intercept are **parameters**, estimated during fitting. The regularization strength is a **hyperparameter**, selected outside that fit using validation. Ridge remains linear in the supplied features. It cannot discover a curved relationship merely by increasing regularization; we must supply suitable transformations or choose another model.

## 3. Why Correlated Features Can Cause Trouble

**Multicollinearity** means one feature is closely related to a combination of other features. To see the extreme case, suppose two already comparable features are identical: $x_1=x_2=x$. Their prediction contribution is

$$
w_1x+w_2x=(w_1+w_2)x.
$$

The coefficient pairs $(2,2)$ and $(20,-16)$ both produce $4x$. Training error alone cannot distinguish them. Their squared coefficient totals, however, are very different:

$$
2^2+2^2=8,\qquad 20^2+(-16)^2=656.
$$

Ridge prefers the balanced pair among pairs with the same sum. It may also shrink that sum when balancing fit against the penalty. With nearly identical features, the predictions are no longer exactly equal, but the same instability can occur.

Correlated features do not automatically imply poor predictions. Predictions can remain accurate where the same feature relationships hold, even while individual coefficients are uncertain. Problems become more visible when new feature combinations differ from those in training.

A large coefficient alone is also not proof of overfitting: its magnitude depends on measurement units. A price coefficient per square meter is numerically different from one per square foot. This is why scaling will be part of our model design.

## 4. From Squared Error to the Ridge Objective

Ordinary least squares minimizes the **sum of squared errors**:

$$
\mathrm{SSE}=\sum_{i=1}^{n}(y_i-\hat y_i)^2.
$$

Squaring prevents positive and negative residuals from canceling and gives large residuals more influence. The mean squared error is $\mathrm{MSE}=\mathrm{SSE}/n$.

In this lesson, we use the following Ridge objective so that our main equations match the companion's scikit-learn estimator:

$$
J(b,\mathbf w)=\sum_{i=1}^{n}\left(y_i-b-\sum_{j=1}^{p}w_jx_{ij}\right)^2
+\alpha\sum_{j=1}^{p}w_j^2,\qquad \alpha\geq0.
$$

Here $\mathbf w$ is the vector containing all $p$ feature coefficients, and $\alpha$ controls the penalty strength. The first term rewards accurate training predictions. The second discourages large coefficients. The intercept $b$ is not penalized: shifting the overall prediction baseline should not incur the same cost as changing feature effects.

The expression $\sum_j w_j^2=\|\mathbf w\|_2^2$ is the **squared L2 norm**. The L2 norm itself takes the square root; Ridge uses its square. A coefficient of 2 contributes 4 to the penalty, while a coefficient of 10 contributes 100.

Ridge minimizes the combined objective, not either term alone. At prediction time, it still uses the ordinary linear equation; no penalty is added to a predicted price.

## 5. Alpha and Lambda: Check the Convention

The original notes use the equally valid formulation

$$
J_{\mathrm{mean}}=\mathrm{MSE}+\lambda\sum_{j=1}^{p}w_j^2.
$$

Multiplying this entire objective by $n$ does not change the minimizing coefficients:

$$
nJ_{\mathrm{mean}}=\mathrm{SSE}+n\lambda\sum_{j=1}^{p}w_j^2.
$$

Therefore, for the same unweighted data, features, and unpenalized intercept, the two formulations agree when

$$
\alpha=n\lambda.
$$

The [scikit-learn Ridge documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) specifies the summed-error convention. Thus `Ridge(alpha=10.0)` in the companion does not mean $\lambda=10$ in the MSE formulation. Other texts may include factors of $1/2$, so always inspect the full objective before comparing penalty values.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>A regularization number has meaning only together with the loss convention, feature scaling, and training data. Do not transfer it blindly between implementations or datasets.</div>

## 6. Comparing Two Candidate Models

Before deriving fitted coefficients, retain a simple illustration from the original notes. Suppose two candidate models use the same target units and feature representation:

| Candidate | MSE | Coefficients | Squared L2 norm |
|---|---:|---|---:|
| A | 10 | 2, 3 | 13 |
| B | 9 | 20, 30 | 1300 |

Using the **mean-error convention** with $\lambda=0.1$:

$$
J_A=10+0.1(13)=11.3,\qquad J_B=9+0.1(1300)=139.
$$

Squared error alone favors B. The penalized objective favors A among these two candidates because B's small error improvement costs a large increase in the penalty.

This comparison explains what the objective values mean. It does not find the optimum over all possible coefficients, and it does not establish that A predicts unseen data better. To see an actual fit, we need to optimize the objective.

## 7. Fitting One Feature: Where Shrinkage Comes From

For one feature, the model is $\hat y_i=b+wx_i$. Let $\bar x$ and $\bar y$ be the training means. For any fixed slope $w$, the best unpenalized intercept is

$$
b=\bar y-w\bar x.
$$

This makes the mean training prediction equal the mean training target. Define centered values $u_i=x_i-\bar x$ and $v_i=y_i-\bar y$. The objective becomes

$$
J(w)=\sum_{i=1}^{n}(v_i-wu_i)^2+\alpha w^2.
$$

A derivative measures how the objective changes when the slope changes. At the minimum, its derivative is zero:

$$
\frac{dJ}{dw}=-2\sum_{i=1}^{n}u_i(v_i-wu_i)+2\alpha w=0.
$$

Collecting the terms containing $w$ gives

$$
w\left(\sum_{i=1}^{n}u_i^2+\alpha\right)=\sum_{i=1}^{n}u_iv_i,
\qquad
w_{\mathrm{ridge}}=\frac{\sum_i u_iv_i}{\sum_i u_i^2+\alpha}.
$$

The numerator measures how the centered feature and target move together. The denominator measures feature variation plus the regularization strength. Ordinary least squares has the same expression without $\alpha$. For a nonconstant feature:

$$
w_{\mathrm{ridge}}=w_{\mathrm{OLS}}\frac{\sum_i u_i^2}{\sum_i u_i^2+\alpha}.
$$

For positive $\alpha$, this factor lies between zero and one. This is literal slope shrinkage in the one-feature case. Multiple correlated features require a joint solution; each coefficient need not follow this simple path.

## 8. Worked Example: Fit and Predict by Hand

Consider three hypothetical houses. Area is measured in thousands of square feet, and price is in thousands of dollars. These values are deliberately simple teaching data.

| House | Area $x_i$ | Price $y_i$ | $u_i=x_i-2$ | $v_i=y_i-200$ | $u_i^2$ | $u_iv_i$ |
|---|---:|---:|---:|---:|---:|---:|
| A | 1 | 100 | -1 | -100 | 1 | 100 |
| B | 2 | 200 | 0 | 0 | 0 | 0 |
| C | 3 | 300 | 1 | 100 | 1 | 100 |
| Total | | | | | 2 | 200 |

The means are $\bar x=2$ and $\bar y=200$. We use the stated units directly, without standardization, to keep the arithmetic visible. Choose $\alpha=2$ in the SSE convention:

$$
w_{\mathrm{ridge}}=\frac{200}{2+2}=50,\qquad b=200-50(2)=100.
$$

Ordinary least squares gives $w_{\mathrm{OLS}}=200/2=100$ and $b=0$. Now compare predictions and residuals:

| Area | Actual price | OLS prediction | Ridge prediction | Ridge residual | Squared residual |
|---|---:|---:|---:|---:|---:|
| 1 | 100 | 100 | 150 | -50 | 2500 |
| 2 | 200 | 200 | 200 | 0 | 0 |
| 3 | 300 | 300 | 250 | 50 | 2500 |

Ridge's SSE is 5000 and its penalty is $2(50^2)=5000$, so its total objective is 10000. Evaluating the OLS coefficients under that same penalized objective gives $0+2(100^2)=20000$. Ridge wins on the objective it was asked to minimize, although OLS has lower training error.

For a new house with $x=2.5$, Ridge predicts $100+50(2.5)=225$, or $225{,}000$. OLS predicts $250{,}000$. We cannot decide which is better without knowing the new house's target or the underlying data-generating relationship.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>Ridge pulls predictions toward the training mean by reducing the slope. This example verifies fitting arithmetic; it does not prove a generalization benefit. If the exact linear relationship continues, shrinkage introduces avoidable error.</div>

## 9. Several Features: The Same Idea in Matrix Form

Matrices let us express the joint fit without a separate equation for every feature. Let $X_c$ be the $n$-by-$p$ matrix of centered training features, and let $\mathbf y_c$ be the vector of centered targets. Then

$$
(X_c^TX_c+\alpha I)\mathbf w=X_c^T\mathbf y_c.
$$

Here $T$ means transpose, $I$ is the $p$-by-$p$ identity matrix, and $X_c^TX_c$ summarizes feature variation and overlap. Adding $\alpha I$ adds $\alpha$ to its diagonal. For $\alpha>0$, this gives a unique coefficient solution even when columns are redundant or there are more features than rows:

$$
\mathbf w=(X_c^TX_c+\alpha I)^{-1}X_c^T\mathbf y_c.
$$

The inverse notation describes the mathematical solution. Numerical software typically solves the system using a suitable decomposition or iterative method rather than explicitly computing an inverse. Restore the intercept with $b=\bar y-\sum_j w_j\bar x_j$.

At $\alpha=0$, the objective becomes ordinary least squares, but redundant columns can make its coefficient solution nonunique. For an unregularized scikit-learn baseline, use `LinearRegression` as the companion does.

The penalty is especially helpful for combinations of features that have very little variation in training: those directions otherwise allow large coefficient changes with little change in error. Ridge controls the overall coefficient norm. With correlated features, an individual coefficient can increase in magnitude or change sign as $\alpha$ changes.

## 10. Geometry and the Bias–Variance Tradeoff

Another way to describe Ridge is to minimize SSE subject to a coefficient budget:

$$
\sum_{j=1}^{p}w_j^2\leq t,
$$

where $t\geq0$ is the allowed squared norm. For two coefficients, this is a disk of radius $\sqrt t$; with more coefficients it is a ball in a higher-dimensional space. The intercept remains outside the budget. Appropriate budgets correspond to penalized solutions, but $t$ is not simply $1/\alpha$.

The round boundary has no corners on the coefficient axes. Ridge therefore does not generally force coefficients to exactly zero. Exact zeros can still occur because of the data or symmetry; Ridge is not a feature-selection procedure.

The statistical tradeoff explains why accepting error can help. **Bias** is systematic error in the average fitted prediction across repeated training samples. **Variance** describes how much predictions change across those samples. Shrinkage introduces bias toward smaller effects, but can reduce variance enough to reduce expected prediction error. Noise that the available features cannot explain remains.

| Strength | Typical behavior | Risk |
|---|---|---|
| Zero | Ordinary least squares objective | Unstable coefficients when data weakly constrain the fit |
| Small positive | Mild restriction | May not control instability enough |
| Moderate | Stronger balance between fit and stability | Must validate whether the balance helps |
| Very large | Coefficients approach zero | Underfitting; predictions approach the training target mean |

For fixed data and representation, an exact Ridge solution has a nonincreasing total coefficient norm as $\alpha$ grows, while training SSE is nondecreasing. Test error has no such monotonic guarantee. Its best value need not be at either extreme.

## 11. Why Feature Scaling Changes the Penalty

Suppose the same area is represented in square feet instead of thousands of square feet. The feature values become 1000 times larger, so the coefficient needed for the same predictions becomes 1000 times smaller. Its squared penalty becomes one million times smaller. Predictions can be identical while the objective's preference changes dramatically.

A common preparation is standardization:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{s_j},
$$

where $\mu_j$ is feature $j$'s training mean and $s_j$ is its training standard deviation. A unit change in $z_j$ then represents one training standard deviation. Comparable numerical scales make a shared penalty strength easier to interpret, but do not make every feature equally informative.

Handle constant columns separately because they carry no variation. For categorical features, use a suitable numerical encoding, such as one-hot indicators, rather than arbitrary category numbers that imply an ordered linear effect. Scaling indicator columns is a modeling choice: standardizing rare indicators changes how strongly their effects are penalized. Evaluate the full representation consistently.

Missing values require an explicit strategy, such as training-fitted imputation. Extreme values deserve inspection because both squared-error fitting and ordinary standardization can be affected by them. Ridge regularization is not robust outlier handling.

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>Fit imputation, encoding, scaling, and Ridge together inside each training fold. Apply those fitted transformations to its validation fold. A pipeline helps keep this sequence consistent.</div>

## 12. Train, Validate, and Use the Model

We now have two separate tasks: fitting coefficients for a fixed $\alpha$, and choosing $\alpha$ for useful predictions. Training error cannot solve the second task because ordinary least squares already minimizes training SSE.

1. Define the target, available prediction-time features, and an evaluation metric. Exclude information created after the outcome is known.
2. Reserve a test set before model selection. Use time-ordered splits for future prediction, or group-aware splits when related rows must stay together.
3. Within the remaining training data, compare candidate alpha values using cross-validation. In each fold, fit preprocessing and coefficients on the fold's training portion and score predictions on its validation portion.
4. Average validation performance across folds and inspect variability. A starting grid might be $0.01,0.1,1,10,100,1000$; these are search candidates, not universal recommendations. Extend the range if results at its edge warrant investigation.
5. Select the configuration using the chosen validation metric, refit its complete pipeline on all development training data, and evaluate once on the reserved test set.
6. For new rows, preserve the same feature definitions and order, apply the fitted preprocessing, and calculate predictions with the fitted intercept and coefficients.

For the companion's pipeline names, the tuning parameter is `model__alpha`. An outer search such as `GridSearchCV` over the entire pipeline allows the scaler to be refitted separately within each fold. Simply scaling all development rows before cross-validation leaks validation information into training.

[Scikit-learn's cross-validation guide](https://scikit-learn.org/stable/modules/cross_validation.html) explains why using test results to select hyperparameters compromises their role as an independent performance estimate. Validation performance guides choices; final test performance assesses the selected procedure.

## 13. Reading the Code Companion Carefully

`03_Ridge_Regression.ipynb` loads `load_diabetes(as_frame=True)`, uses its features and numerical target, and creates an 80/20 split with `random_state=42`. It fits two pipelines: `StandardScaler` followed by `LinearRegression`, and `StandardScaler` followed by `Ridge(alpha=10.0)`.

The companion then compares test MAE, RMSE, and $R^2$, inspects coefficients, and loops through alpha values from 0.01 to 1000. This connects directly to our theory:

| Companion item | How to interpret it |
|---|---|
| `scaler` inside each pipeline | Scaling is learned from the rows supplied to `fit` |
| `coef_` | Coefficients in the scaled feature representation |
| `absolute_shrinkage` | OLS coefficient magnitude minus Ridge magnitude for each feature |
| `coefficient_norm` | L2 norm, the square root of the squared norm used in the penalty |
| Alpha sweep | Illustration of how predictions and coefficients vary with the penalty |

A negative `absolute_shrinkage` entry is possible: total norm shrinkage does not require every correlated feature's coefficient magnitude to decrease.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Keep the test set independent</strong><br>The companion's sweep evaluates each alpha on the same test set. Treat it as an illustration. Selecting its best test score would turn that test set into validation data; use training-only cross-validation for tuning.</div>

The fixed alpha of 10 is an example setting, not an established optimum. These notes describe the companion's source code and do not claim measured scores or that Ridge wins its comparison.

## 14. Evaluating Predictions

Evaluate prediction error on held-out rows without adding the training penalty. Let $m$ be the number of evaluation rows and $e_i=y_i-\hat y_i$ their residuals:

$$
\mathrm{MAE}=\frac{1}{m}\sum_{i=1}^{m}|e_i|,\qquad
\mathrm{MSE}=\frac{1}{m}\sum_{i=1}^{m}e_i^2,\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}.
$$

MAE describes average absolute error in target units. RMSE also has target units but gives large errors more influence. MSE has squared target units. Lower values are better when comparing the same evaluation rows and target representation.

For nonconstant evaluation targets:

$$
R^2=1-\frac{\sum_{i=1}^{m}(y_i-\hat y_i)^2}{\sum_{i=1}^{m}(y_i-\bar y_{\mathrm{eval}})^2},
$$

where $\bar y_{\mathrm{eval}}$ is their mean. A perfect fit has $R^2=1$; zero matches the constant evaluation-mean reference in this formula, and negative values are worse. It is not a percentage accuracy score. Constant evaluation targets make this mathematical ratio undefined. For an actual deployable baseline, also compare against a predictor fitted to the training target mean.

Aggregate scores can hide problems. Inspect residuals against predictions and important features: curvature can suggest missing nonlinear structure, while widening residual spread suggests unequal error variability. Evaluate relevant groups and time periods as well. A small score improvement on one split is weak evidence of a reliable advantage.

## 15. Interpreting Coefficients Without Overclaiming

For raw features, $w_j$ is the change in predicted target per one-unit increase in feature $j$, holding the other represented features fixed. For standardized features, it is the change per one training standard deviation. The intercept is then the prediction at the training means of all standardized features.

To recover a raw-unit coefficient from a standardized coefficient $w_j$, divide by its training scale: $\beta_j=w_j/s_j$. If $b_z$ is the intercept in standardized coordinates, the raw-coordinate intercept is

$$
b_{\mathrm{raw}}=b_z-\sum_{j=1}^{p}\frac{w_j\mu_j}{s_j}.
$$

These formulas preserve predictions; they do not refit the model. They assume the target itself was not transformed.

Coefficient magnitude is not a complete measure of importance. Correlated features can share signal, regularization deliberately biases coefficients, and changing the representation can redistribute weights. A small coefficient does not prove a feature has no predictive value.

Holding one feature fixed while changing another may describe an unrealistic house if the features are tightly related. And a predictive association is not a causal effect: the coefficient for floor area does not establish how much a renovation would change a sale price. Ridge also does not automatically supply valid uncertainty intervals or ordinary least-squares significance tests.

## 16. Design Choices and Nearby Alternatives

| Choice | What it changes | How to decide |
|---|---|---|
| `alpha` | Strength of the coefficient penalty | Training-only validation on the chosen metric |
| Intercept | Allows a baseline independent of feature effects | Usually retain it; omit only with a justified zero-baseline or centering setup |
| Feature representation and scaling | Which relationships can be learned and how they are penalized | Domain reasoning plus validation of the full pipeline |
| Numerical solver | How the same fitting problem is computed | Consider data size, sparsity, and convergence |
| Iterative tolerance and iteration limit | Numerical accuracy and stopping | Check convergence; these are not substitutes for alpha tuning |

Ridge has one learned coefficient per supplied feature, so ordinary dense prediction needs roughly $p$ multiplications per row. Training cost depends on the solver. A direct approach that builds a feature cross-product matrix can take roughly $O(np^2+p^3)$ work and $O(p^2)$ extra storage, while other decompositions and iterative methods have different costs. A mathematically compact solution does not imply negligible cost for a very large feature matrix.

| Method | Main difference from Ridge | Useful comparison |
|---|---|---|
| Ordinary least squares | No coefficient penalty | Baseline when the fit is stable and regularization adds little benefit |
| Lasso | Penalizes the sum of absolute coefficient values | Can produce exact zeros when a sparse model is desired |
| Elastic Net | Combines absolute-value and squared penalties | Balances sparsity with handling correlated predictors |
| Polynomial features plus Ridge | Supplies powers or interactions before fitting | Can represent curves while regularizing the expanded coefficients |
| Trees or forests | Learn split-based relationships | Useful alternatives for nonlinear patterns and interactions |

These choices should be compared with appropriate preprocessing on the same validation design. Lasso's sparsity does not guarantee selection of the true causal features, and nonlinear models do not automatically outperform a well-specified linear model.

## 17. Strengths, Assumptions, and Failure Modes

Ridge is a useful candidate when many features may contribute modest signal, ordinary least squares is unstable, or correlated predictors make the individual coefficients difficult to estimate. It retains a compact linear prediction rule and can fit a unique regularized solution even with redundant features.

Its usefulness depends on the feature representation capturing enough of the relationship and on training data being relevant to future predictions. Independent, representative rows support ordinary random-split evaluation; time dependence or repeated subjects require a more appropriate split. Normally distributed errors are not required to compute a Ridge fit. Stronger assumptions would be needed for particular inferential claims.

| Failure mode | Why Ridge alone does not fix it | Practical response |
|---|---|---|
| Strong unrepresented nonlinear structure | Shrinkage does not create new features | Consider transformations, interactions, or another model |
| Outliers or bad measurements | Squared error can give them large influence | Investigate data quality and consider robust methods |
| Excessive penalty | Useful effects are suppressed | Revisit validation and the alpha range |
| Leakage | Unavailable future information makes evaluation optimistic | Audit features and fit preprocessing within folds |
| Distribution shift | Future relationships may differ from training | Use realistic evaluation and monitor later performance |
| Need for a small feature subset | Ridge generally retains nonzero coefficients | Compare sparse methods or validated feature selection |

Ridge can extrapolate beyond the training target range because its prediction rule is linear, but that is a mathematical ability, not evidence that such predictions are reliable. Likewise, a unique solution is not a guarantee of accurate predictions or interpretable causal effects.

## 18. Recap

Ridge fits the same prediction equation as Linear Regression while penalizing the squared coefficient norm. The penalty discourages unstable solutions, especially when features overlap, at the cost of extra training error and bias.

The most important choices are the feature representation, scaling, regularization strength, and validation design. Match the penalty convention to the implementation, fit preprocessing inside each training fold, and assess final predictions on an independent test set.

Remember the worked example: a smaller penalized objective did not prove a better prediction for the next house. Regularization is a preference that must earn its place through realistic validation.